#1. Instalación de librerias


In [2]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 17.0 MB/s eta 0:00:00


#2. Importación de librerias

In [3]:
import os
import requests
import json
import pandas as pd
from datetime import datetime
from pymongo import MongoClient

#3.Conexión a MongoDB y creación de colección

In [4]:
# Leer las credenciales desde el archivo JSON
with open('config.json', 'r') as f:
    config = json.load(f)

# Configuración de MongoDB
uri = config["mongo_uri"]
client = MongoClient(uri)
db = client["crypto_analysis"]

# Fecha actual para nombrar archivo y colección
today_str = datetime.today().strftime("%d_%m_%Y")
collection_name = f"memecoins_prices_data_{today_str}"
collection = db[collection_name]

#4.Descarga de datos CoinGecko

## 4.1 Crear carpeta para CSV

In [5]:
# Crear carpeta para el CSV si no existe
os.makedirs("memecoin_prices_csv", exist_ok=True)

## 4.2 Definir indentificadores y descargar los datos

In [6]:
# Identificadores de CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

all_data = []

# Descargar y combinar datos
for name, coingecko_id in coins.items():
    print(f"Descargando datos para {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/market_chart'
    params = {
        'vs_currency': 'usd',
        'days': '365',
        'interval': 'daily'
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error al obtener datos para {name}: {response.status_code}")
        continue

    data = response.json()

    # Crear DataFrames
    prices_df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
    volumes_df = pd.DataFrame(data['total_volumes'], columns=['timestamp', 'volume'])
    market_caps_df = pd.DataFrame(data['market_caps'], columns=['timestamp', 'market_cap'])

    # Convertir a fechas
    for df in [prices_df, volumes_df, market_caps_df]:
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date
        df.drop(columns='timestamp', inplace=True)

    # Merge y formateo final
    merged_df = prices_df.merge(volumes_df, on='date').merge(market_caps_df, on='date')
    merged_df['coin'] = name
    merged_df = merged_df[['date', 'coin', 'price', 'volume', 'market_cap']]
    all_data.append(merged_df)

Descargando datos para PEPE...
Descargando datos para DOGE...
Descargando datos para SHIBA...


###4.3 Concatenar los datos y guardarlos en CSV

In [7]:
# Obtener la fecha actual para el nombre del archivo
today_str = datetime.today().strftime("%d_%m_%y")

# Concatenar todo
if not all_data:
    print("⚠️ No se descargaron datos.")
else:
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df['date'] = pd.to_datetime(combined_df['date'])

    # Guardar CSV único
    csv_path = f"memecoin_prices_csv/memecoins_prices_data_{today_str}.csv"
    combined_df.to_csv(csv_path, index=False, sep=';')
    print(f"📁 CSV guardado: {csv_path}")

📁 CSV guardado: memecoin_prices_csv/memecoins_prices_data_05_06_25.csv


## 4.4 Subir los datos a MongoDB

In [8]:
# Nombre de la colección
collection_name = f"memecoins_prices_data_{today_str}"
collection = db[collection_name]

# Verificar si la colección ya existe y eliminarla si es necesario
if collection_name in db.list_collection_names():
    db.drop_collection(collection_name)
    print(f"Colección '{collection_name}' eliminada previamente.")

# Subir a MongoDB
collection.insert_many(combined_df.to_dict("records"))
print(f"Datos subidos a MongoDB: colección '{collection_name}'")

✅ Datos subidos a MongoDB: colección 'memecoins_prices_data_05_06_25'
